# Intermediate Pandas

**Estimated time:** 45–60 minutes  
**Prerequisites:** Basic Python, familiarity with DataFrames and Series

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Multi-level indexing and `xs` slicing |
| 2 | `groupby` — transform, filter, and agg together |
| 3 | Merging and joining DataFrames |
| 4 | Pivot tables and `melt` (reshaping) |
| 5 | String methods (`.str`) and categorical dtype |
| 6 | Time series — resampling and rolling windows |
| 7 | `apply` and vectorized alternatives |
| 8 | `pipe` for method chaining |

---

### Quick Reference

```
df.groupby(keys).agg({'col': ['mean','std']})
df.merge(other, on='key', how='left')
df.pivot_table(values, index, columns, aggfunc)
df.melt(id_vars, value_vars, var_name, value_name)
df.resample('M').sum()          # requires DatetimeIndex
df['col'].rolling(n).mean()
```

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
print('pandas', pd.__version__)

---
## Section 1 — Multi-Level Indexing

A **MultiIndex** lets you represent hierarchical data (e.g. company → year → quarter) without repeating columns.

Key methods:
- `set_index([col1, col2])` — build the index
- `df.loc[(level0_val, level1_val)]` — label-based selection
- `df.xs(key, level=name)` — cross-section across any level
- `df.reset_index()` — flatten back to regular columns

In [ ]:
# Build a small claims dataset with hierarchical structure
np.random.seed(42)
companies = ['Alpha', 'Beta', 'Gamma']
lines = ['Auto', 'Property', 'Liability']
years = [2021, 2022, 2023]

rows = []
for co in companies:
    for ln in lines:
        for yr in years:
            rows.append({
                'company': co,
                'line': ln,
                'year': yr,
                'premium': np.random.randint(500_000, 5_000_000),
                'claims': np.random.randint(100_000, 3_000_000),
            })

df = pd.DataFrame(rows)
df['loss_ratio'] = df['claims'] / df['premium']
print(df.shape)
df.head(6)

In [ ]:
# Set a two-level index
dfm = df.set_index(['company', 'line']).sort_index()
dfm.head(9)

In [ ]:
# Accessing with .loc — tuple selects both levels
dfm.loc[('Alpha', 'Auto')]

In [ ]:
# .xs lets you slice on an inner level without specifying the outer
# Get all 'Auto' rows across every company
dfm.xs('Auto', level='line')

In [ ]:
# EXERCISE 1: Use .xs to retrieve all rows for year 2022
# Hint: first you need 'year' in the index — try adding it to set_index

dfm3 = df.set_index(['company', 'line', 'year']).sort_index()
# YOUR CODE HERE
# result = dfm3.xs(...)
# result

---
## Section 2 — GroupBy: agg, transform, filter

Three distinct use cases:

| Method | Returns | Use when |
|--------|---------|----------|
| `.agg()` | One row per group (reduced) | Summary statistics |
| `.transform()` | Same shape as input | Add group-level stats as columns |
| `.filter()` | Subset of original rows | Keep/drop entire groups |

Named aggregations with `pd.NamedAgg` keep output column names clean.

In [ ]:
# Multi-function agg — returns a MultiIndex column
summary = df.groupby('line').agg(
    total_premium=('premium', 'sum'),
    total_claims=('claims', 'sum'),
    avg_lr=('loss_ratio', 'mean'),
    std_lr=('loss_ratio', 'std'),
    n=('year', 'count'),
)
summary

In [ ]:
# transform: add a column showing each row's deviation from its group mean
df['line_avg_lr'] = df.groupby('line')['loss_ratio'].transform('mean')
df['lr_vs_line_avg'] = df['loss_ratio'] - df['line_avg_lr']

# Companies where LR is worse than the line average show positive deviation
df[['company', 'line', 'year', 'loss_ratio', 'line_avg_lr', 'lr_vs_line_avg']].head(9)

In [ ]:
# filter: keep only lines where the average loss ratio > 0.55
high_lr = df.groupby('line').filter(lambda g: g['loss_ratio'].mean() > 0.55)
print('Lines kept:', high_lr['line'].unique())
high_lr.shape

In [ ]:
# EXERCISE 2:
# a) For each (company, year), compute total premium and total claims
# b) Add a column 'rank_within_year' that ranks companies by total premium within each year
#    Hint: groupby('year')['total_premium'].rank(ascending=False)

# YOUR CODE HERE

---
## Section 3 — Merging and Joining

```
pd.merge(left, right, on=key, how='left|right|inner|outer')
df.join(other, on=key)   # index-based shorthand
```

| Join type | Keeps |
|-----------|-------|
| `inner`   | Only rows with matches in **both** |
| `left`    | All left rows; NaN where no right match |
| `right`   | All right rows; NaN where no left match |
| `outer`   | All rows from both sides |

**Diagnostic tip:** check `df.merge(..., indicator=True)['_merge'].value_counts()` to spot unmatched rows.

In [ ]:
# Company metadata table
meta = pd.DataFrame({
    'company': ['Alpha', 'Beta', 'Gamma', 'Delta'],  # Delta has no claims data
    'region': ['Northeast', 'Southeast', 'Midwest', 'West'],
    'founded': [1982, 1995, 2003, 2010],
})

# Aggregate df to company level first
co_summary = df.groupby('company', as_index=False).agg(
    total_premium=('premium', 'sum'),
    total_claims=('claims', 'sum'),
)
co_summary['combined_lr'] = co_summary['total_claims'] / co_summary['total_premium']
co_summary

In [ ]:
# Inner join — only Alpha, Beta, Gamma (Delta has no claims data)
inner = pd.merge(co_summary, meta, on='company', how='inner')
inner

In [ ]:
# Outer join with indicator — Delta appears with NaN financials
outer = pd.merge(co_summary, meta, on='company', how='outer', indicator=True)
outer

In [ ]:
# Diagnostic — see which rows matched
outer['_merge'].value_counts()

In [ ]:
# EXERCISE 3:
# Rate table: different expense load by line of business
rates = pd.DataFrame({
    'line': ['Auto', 'Property', 'Liability'],
    'expense_load': [0.30, 0.35, 0.25],
})

# Join rates onto df and create a new column 'pure_premium'
# pure_premium = premium * (1 - expense_load)
# YOUR CODE HERE

---
## Section 4 — Reshaping: pivot_table and melt

`pivot_table` → **wide format** (rows × columns matrix)  
`melt` → **long format** (one observation per row)

These are inverses of each other. Long format is better for plotting; wide format is better for reports.

In [ ]:
# Pivot: rows = line, columns = year, values = average loss ratio
pivot = df.pivot_table(
    values='loss_ratio',
    index='line',
    columns='year',
    aggfunc='mean',
)
pivot.round(3)

In [ ]:
# Add margins (row/column totals — uses aggfunc, so mean here)
pivot_m = df.pivot_table(
    values='loss_ratio',
    index='line',
    columns='year',
    aggfunc='mean',
    margins=True,
    margins_name='Overall',
)
pivot_m.round(3)

In [ ]:
# melt: convert the wide pivot back to long format
pivot_reset = pivot.reset_index()  # bring 'line' back as a column
long = pivot_reset.melt(
    id_vars='line',
    var_name='year',
    value_name='avg_loss_ratio',
)
long.sort_values(['line', 'year']).head(10)

In [ ]:
# EXERCISE 4:
# Create a pivot table showing TOTAL CLAIMS (sum) with
#   rows = company, columns = line, values = claims
# Then find which (company, line) cell has the highest claims
# YOUR CODE HERE

---
## Section 5 — String Methods and Categorical Dtype

`.str` accessor vectorizes string operations without loops.

| Method | Purpose |
|--------|---------|
| `.str.upper()` / `.lower()` | Case conversion |
| `.str.contains(pat, regex=True)` | Boolean mask |
| `.str.extract(pattern)` | Capture groups → columns |
| `.str.split(sep, expand=True)` | Split into columns |
| `.str.replace(pat, repl)` | Substitution |

**Categorical dtype** stores a column as integer codes + lookup table — saves memory and speeds up `groupby` on low-cardinality columns.

In [ ]:
# Work with a messy text column
claims_notes = pd.DataFrame({
    'claim_id': range(1, 9),
    'description': [
        'AUTO - rear-end collision, bodily injury',
        'PROPERTY - fire damage, total loss',
        'auto - side-swipe, minor',
        'LIABILITY - slip and fall',
        'Property - water damage',
        'AUTO - theft, comprehensive',
        'liability - dog bite',
        'AUTO - hail damage, comprehensive',
    ]
})

# Normalize to title case and extract line of business
claims_notes['desc_clean'] = claims_notes['description'].str.title()
claims_notes['lob'] = claims_notes['description'].str.extract(r'^([A-Za-z]+)')[0].str.title()
claims_notes

In [ ]:
# Filter to comprehensive claims
comp = claims_notes[claims_notes['description'].str.contains('comprehensive', case=False)]
comp

In [ ]:
# Convert 'line' in main df to categorical — compare memory usage
df['line_obj'] = df['line']  # object dtype copy
df['line_cat'] = df['line'].astype('category')

print(f"object dtype:      {df['line_obj'].memory_usage(deep=True):,} bytes")
print(f"category dtype:    {df['line_cat'].memory_usage(deep=True):,} bytes")

# Categories are ordered — useful for sorting
cat_type = pd.CategoricalDtype(['Auto', 'Property', 'Liability'], ordered=True)
df['line_ordered'] = df['line'].astype(cat_type)
df.sort_values('line_ordered')[['company','line_ordered','year']].head(10)

In [ ]:
# EXERCISE 5:
# Given descriptions below, extract both the lob AND the specific peril
# (the text after the dash) into separate columns using .str.extract
test = pd.Series([
    'AUTO - rear-end collision',
    'PROPERTY - fire damage',
    'LIABILITY - slip and fall',
])
# Hint: pattern r'^([A-Z]+) - (.+)'
# YOUR CODE HERE

---
## Section 6 — Time Series: Resampling and Rolling Windows

Requires a **DatetimeIndex**. Convert with `pd.to_datetime` and `set_index`.

| Operation | Syntax | What it does |
|-----------|--------|--------------|
| Resample | `df.resample('M').sum()` | Aggregate to calendar period |
| Rolling | `df.rolling(n).mean()` | Trailing n-period average |
| Expanding | `df.expanding().sum()` | Cumulative sum from start |
| Shift | `df.shift(n)` | Lag values by n periods |

**Common resample aliases:** `'D'` daily, `'W'` weekly, `'ME'` month-end, `'QE'` quarter-end, `'YE'` year-end

In [ ]:
# Simulate daily claims payments over 2 years
np.random.seed(7)
dates = pd.date_range('2022-01-01', '2023-12-31', freq='D')
ts = pd.DataFrame({
    'date': dates,
    'paid_claims': np.random.gamma(shape=2, scale=50_000, size=len(dates)).astype(int),
    'new_claims': np.random.poisson(lam=15, size=len(dates)),
}).set_index('date')

print(ts.shape)
ts.head()

In [ ]:
# Monthly totals
monthly = ts.resample('ME').sum()
monthly.head(6)

In [ ]:
# 30-day rolling average on daily data
ts['rolling_30d_paid'] = ts['paid_claims'].rolling(30).mean()

# Expanding (cumulative) total of new claims
ts['cumulative_new'] = ts['new_claims'].expanding().sum()

ts.head(35).tail(10)

In [ ]:
# Month-over-month change in paid claims
monthly['prev_month'] = monthly['paid_claims'].shift(1)
monthly['pct_change'] = monthly['paid_claims'].pct_change() * 100
monthly.round(1).head(8)

In [ ]:
# EXERCISE 6:
# a) Resample ts to quarterly totals (both columns)
# b) Compute a 90-day rolling MAX on paid_claims (different from mean — why might you care?)
# YOUR CODE HERE

---
## Section 7 — apply vs. Vectorized Operations

`apply` is flexible but slow — it runs Python in a loop. Prefer vectorized alternatives:

| Task | apply (slow) | Vectorized (fast) |
|------|-------------|-------------------|
| Conditional value | `apply(lambda r: ...)` | `np.where` / `pd.cut` |
| Math on column | `apply(math.sqrt)` | `np.sqrt(df['col'])` |
| String format | `apply(lambda x: f'{x:.1%}')` | `.map('{:.1%}'.format)` |
| Row-wise logic | `apply(func, axis=1)` | multiple column ops |

Use `apply` when no vectorized option exists (e.g. complex multi-column row logic or calling an external API).

In [ ]:
# BAD: apply for a simple condition
# df['flag'] = df['loss_ratio'].apply(lambda x: 'High' if x > 0.7 else 'OK')

# GOOD: np.where
df['flag'] = np.where(df['loss_ratio'] > 0.7, 'High', 'OK')

# GOOD: pd.cut for bins
df['lr_band'] = pd.cut(
    df['loss_ratio'],
    bins=[0, 0.5, 0.7, 0.9, 1.5],
    labels=['Low', 'Moderate', 'High', 'Extreme']
)

df[['company', 'line', 'year', 'loss_ratio', 'flag', 'lr_band']].head(10)

In [ ]:
# Timing comparison on a larger dataset
big = pd.DataFrame({'x': np.random.rand(200_000)})

%timeit big['x'].apply(lambda v: v ** 2)
%timeit big['x'] ** 2

In [ ]:
# Legitimate apply: complex row-level logic with multiple columns
def classify_risk(row):
    """Combine loss ratio and premium size into a risk tier."""
    if row['loss_ratio'] > 0.80 and row['premium'] < 1_000_000:
        return 'Concern'
    elif row['loss_ratio'] > 0.80:
        return 'Monitor'
    else:
        return 'Acceptable'

df['risk_tier'] = df.apply(classify_risk, axis=1)
df['risk_tier'].value_counts()

In [ ]:
# EXERCISE 7:
# Replace this apply with a vectorized equivalent using np.select or np.where
# tier_apply = df.apply(
#     lambda r: 'Large' if r['premium'] > 3_000_000
#               else ('Medium' if r['premium'] > 1_500_000 else 'Small'),
#     axis=1
# )
# YOUR CODE HERE — create df['size_tier'] without using apply

---
## Section 8 — Method Chaining with pipe

`pipe` lets you slot custom functions into a chain, keeping the readable top-to-bottom flow of `.groupby().agg().reset_index()` without intermediate variables.

```python
result = (
    raw_df
    .pipe(clean_columns)
    .pipe(add_features)
    .query('loss_ratio < 2')
    .groupby('line')
    .agg(...)
)
```

In [ ]:
def add_lr_flag(df, threshold=0.70):
    """Add a high-LR indicator column."""
    df = df.copy()
    df['high_lr'] = df['loss_ratio'] > threshold
    return df

def add_premium_tier(df):
    """Bucket premium into Small/Medium/Large."""
    df = df.copy()
    df['size'] = pd.cut(
        df['premium'],
        bins=[0, 1_500_000, 3_000_000, float('inf')],
        labels=['Small', 'Medium', 'Large'],
    )
    return df

result = (
    df[['company', 'line', 'year', 'premium', 'claims', 'loss_ratio']]
    .pipe(add_lr_flag, threshold=0.65)  # pass extra args after df
    .pipe(add_premium_tier)
    .query("year == 2023")
    .groupby(['line', 'size'], observed=True)
    .agg(count=('company','count'), avg_lr=('loss_ratio','mean'))
    .round(3)
)
result

In [ ]:
# EXERCISE 8 (capstone):
# Write a function normalize_lr(df) that:
#   - Computes z-score of loss_ratio within each line:  (lr - mean) / std
#   - Adds column 'lr_zscore'
# Then build a pipe chain that:
#   1. Applies normalize_lr
#   2. Filters to |lr_zscore| > 1 (outliers)
#   3. Sorts by lr_zscore descending
# YOUR CODE HERE

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| MultiIndex | Use `.xs(key, level=name)` for inner-level slices |
| groupby | `agg` shrinks, `transform` preserves shape, `filter` drops groups |
| merge | Always check with `indicator=True`; unmatched rows become NaN |
| reshape | `pivot_table` → wide; `melt` → long; they're inverses |
| strings | `.str.extract(r'pattern')` for structured text parsing |
| category | Low-cardinality columns → `astype('category')` saves memory |
| time series | `resample` aggregates; `rolling`/`expanding` slide windows |
| apply | Last resort — prefer `np.where`, `pd.cut`, `np.select` |
| pipe | Keeps multi-step transforms readable as a single chain |